# Deep Learning Intro (PyTorch)

In [ ]:
# Day 32 — Deep Learning Intro (PyTorch)
# Classical ML (scikit-learn, XGBoost) is king for tabular data.
# Deep Learning (PyTorch) is what powers computer vision, NLP, and ALL LLMs!
#
# Even if you don't train LLMs from scratch as an MLOps/LLMOps engineer,
# you WILL:
#   - Manage GPU containers (CUDA, VRAM, device placement)
#   - Load/serve PyTorch model weights (.pt / .pth / state_dict)
#   - Debug tensor shape errors and OOM (Out Of Memory) crashes
#
# Today we'll cover:
#   1. Tensors vs NumPy arrays
#   2. Autograd (automatic differentiation)
#   3. Building a Neural Net with nn.Module
#   4. Loss functions, Optimizers, and the Training Loop
#   5. GPU basics (.to('cuda')) & saving/loading state_dict
#
# Install: pip install torch

### 1. Tensors vs NumPy Arrays

In [ ]:
# A PyTorch Tensor is just like a NumPy ndarray, EXCEPT:
#   1. It can run on a GPU for massive parallelism
#   2. It can track gradients automatically (autograd)

import torch
import numpy as np

# Creating tensors
t1 = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("Tensor:\n", t1)
print("Shape:", t1.shape, "| Dtype:", t1.dtype, "| Device:", t1.device)

# Converting between NumPy and PyTorch
np_arr = np.array([10.0, 20.0, 30.0], dtype=np.float32)
t_from_np = torch.from_numpy(np_arr)
print("From NumPy:", t_from_np)

# GPU Check (critical in MLOps!)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
# Move tensor to GPU (if available): t1 = t1.to(device)

### 2. Autograd — How Neural Networks Learn

In [ ]:
# Neural networks learn by computing gradients (derivatives of loss wrt weights).
# PyTorch tracks operations when requires_grad=True.

import torch

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x + 1    # dy/dx = 2x + 2 -> at x=3, gradient = 8

y.backward()              # compute gradient
print("y value:", y.item())
print("dy/dx at x=3:", x.grad.item())

# During inference (serving), we disable gradient tracking to save memory:
with torch.no_grad():
    z = x * 5
    print("Inference without grad:", z.item())

### 3. Building a Neural Network (`nn.Module`) & Training Loop

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define the model architecture
class ServerHealthNet(nn.Module):
    def __init__(self, input_features=4):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_features, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2)    # 2 classes: 0=healthy, 1=critical
        )

    def forward(self, x):
        return self.network(x)

model = ServerHealthNet(input_features=4)
print(model)

# Count trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params}")

# Synthetic data: [cpu, memory, disk, errors] normalized 0-1
torch.manual_seed(42)
X = torch.rand(200, 4)
y = (X[:, 0] + X[:, 1] + X[:, 3] > 1.6).long()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Standard PyTorch Training Loop
for epoch in range(1, 51):
    model.train()
    optimizer.zero_grad()          # 1. Clear old gradients
    outputs = model(X)             # 2. Forward pass
    loss = criterion(outputs, y)   # 3. Compute loss
    loss.backward()                # 4. Backpropagation
    optimizer.step()               # 5. Update weights

    if epoch % 10 == 0:
        preds = outputs.argmax(dim=1)
        acc = (preds == y).float().mean().item()
        print(f"Epoch {epoch:2d} | Loss: {loss.item():.4f} | Acc: {acc:.2%}")

### 4. Saving & Loading Models (`state_dict`)

In [ ]:
# Best practice in PyTorch: save the state_dict (weights), NOT the whole pickled class.

torch.save(model.state_dict(), "server_health_net.pt")
print("Saved weights to server_health_net.pt")

# Loading for inference
loaded_model = ServerHealthNet(input_features=4)
loaded_model.load_state_dict(torch.load("server_health_net.pt", weights_only=True))
loaded_model.eval()    # sets dropout/batchnorm to evaluation mode!

sample = torch.tensor([[0.90, 0.85, 0.70, 0.80]])
with torch.no_grad():
    logits = loaded_model(sample)
    pred = logits.argmax(dim=1).item()
print("Sample prediction:", "critical" if pred == 1 else "healthy")

In [ ]:
# Practice Questions:
#
# 1. Why do we call optimizer.zero_grad() at the start of every training step?
# 2. What is the difference between model.train() and model.eval()?
# 3. Why should you use `with torch.no_grad():` during API inference?
# 4. When should you stick to XGBoost/scikit-learn instead of PyTorch?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: Neural Net vs Scikit-learn Comparison
# Train both a RandomForestClassifier and a PyTorch nn.Module on the same
# server health dataset (1000 rows). Compare accuracy and inference latency.

# Exercise 2: Model Complexity & Memory Inspector
# Write a helper function `inspect_pytorch_model(model)` that prints:
# - Layer names and output shapes
# - Total parameter count
# - Estimated memory size in MB (params * 4 bytes for float32)

# Exercise 3: GPU Awareness Check Script
# Write a pre-flight check script for a Docker container that prints:
# - PyTorch version
# - CUDA availability & GPU name/VRAM (if available), else falls back cleanly to CPU